# 07 Train GraphEdgeClassifier - Plain

Notebook ini sudah disesuaikan untuk split baru:
- `train_1_50.csv` untuk training
- `val_temporal.csv` untuk validation / threshold tuning
- `test_temporal.csv` untuk final evaluation

Validation dipakai untuk memilih best epoch dan threshold. Test hanya dipakai sekali di akhir.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cpu')

In [2]:
# =========================
# PATH CONFIG
# =========================

DATA_DIR = Path("../data/processed")
OUTPUT_METRICS_DIR = Path("../outputs/metrics")
OUTPUT_MODELS_DIR = Path("../outputs/models")

OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_1_50.csv"
VAL_PATH   = DATA_DIR / "val_temporal.csv"
TEST_PATH  = DATA_DIR / "test_temporal.csv"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(p, "exists=", p.exists())


..\data\processed\train_1_50.csv exists= True
..\data\processed\val_temporal.csv exists= True
..\data\processed\test_temporal.csv exists= True


In [3]:
# =========================
# LOAD NEW TEMPORAL SPLIT
# =========================

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
val_df   = pd.read_csv(VAL_PATH, low_memory=False)
test_df  = pd.read_csv(TEST_PATH, low_memory=False)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("Columns:")
print(train_df.columns.tolist())


Train: (35802, 25)
Val  : (149079, 25)
Test : (298160, 25)
Columns:
['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'timestamp_dt', 'mint_timestamp_dt', 'month', 'is_wash_trading', 'rule_self_trade', 'rule_seller_buyback', 'rule_multi_hop_cycle', 'rule_high_pair_count', 'wash_score', 'confidence_category', 'label_final']


In [4]:
# =========================
# COLUMN CONFIG
# =========================

SOURCE_COL = "from_address"
TARGET_COL = "to_address"
LABEL_COL = "label_final"
TIMESTAMP_COL = "timestamp"

required_cols = [SOURCE_COL, TARGET_COL, LABEL_COL, TIMESTAMP_COL]
missing = [c for c in required_cols if c not in train_df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(name, df[LABEL_COL].value_counts(dropna=False).to_dict())
    print(name, "fraud_rate=", df[LABEL_COL].mean())


Train {0: 35100, 1: 702}
Train fraud_rate= 0.0196078431372549
Val {0: 149027, 1: 52}
Val fraud_rate= 0.0003488083499352692
Test {0: 298096, 1: 64}
Test fraud_rate= 0.00021464985242822645


In [5]:
# =========================
# SORT TEMPORALLY
# =========================

for df in [train_df, val_df, test_df]:
    df[TIMESTAMP_COL] = pd.to_numeric(df[TIMESTAMP_COL], errors="coerce")

train_df = train_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
val_df   = val_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
test_df  = test_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

print("Train timestamp:", train_df[TIMESTAMP_COL].min(), "->", train_df[TIMESTAMP_COL].max())
print("Val timestamp  :", val_df[TIMESTAMP_COL].min(), "->", val_df[TIMESTAMP_COL].max())
print("Test timestamp :", test_df[TIMESTAMP_COL].min(), "->", test_df[TIMESTAMP_COL].max())


Train timestamp: 1622507092 -> 1629695254
Val timestamp  : 1629695273 -> 1629978053
Test timestamp : 1629978053 -> 1630454395


In [6]:
# =========================
# NODE MAPPING
# =========================
# Mapping dibuat dari train+val+test supaya wallet baru di validation/test tetap punya ID.
# Ini hanya memakai daftar wallet, bukan label/future feature untuk training.

all_nodes = pd.concat([
    train_df[SOURCE_COL], train_df[TARGET_COL],
    val_df[SOURCE_COL], val_df[TARGET_COL],
    test_df[SOURCE_COL], test_df[TARGET_COL],
], axis=0).astype(str).fillna("UNKNOWN_WALLET").unique()

node_to_id = {node: idx for idx, node in enumerate(all_nodes)}
num_nodes = len(node_to_id)

print("Num nodes:", num_nodes)


def map_nodes(df):
    src = df[SOURCE_COL].astype(str).fillna("UNKNOWN_WALLET").map(node_to_id)
    dst = df[TARGET_COL].astype(str).fillna("UNKNOWN_WALLET").map(node_to_id)
    if src.isna().any() or dst.isna().any():
        raise ValueError(f"Ada node yang tidak ada di mapping. missing_src={src.isna().sum()}, missing_dst={dst.isna().sum()}")
    return src.astype(np.int64).values, dst.astype(np.int64).values


Num nodes: 126505


In [7]:
# =========================
# FEATURE SELECTION
# =========================

leakage_cols = {
    "is_wash_trading",
    "label_final",
    "rule_self_trade",
    "rule_seller_buyback",
    "rule_multi_hop_cycle",
    "rule_high_pair_count",
    "wash_score",
    "confidence_category",
}

id_time_cols = {
    SOURCE_COL,
    TARGET_COL,
    LABEL_COL,
    TIMESTAMP_COL,
    "transaction_hash",
    "nft_address",
    "token_id",
    "timestamp_dt",
    "mint_timestamp_dt",
}

exclude_cols = leakage_cols | id_time_cols

numeric_cols = train_df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

print("Num feature cols:", len(feature_cols))
print(feature_cols)

leaked = [c for c in feature_cols if c in leakage_cols]
assert len(leaked) == 0, f"Leakage masih masuk: {leaked}"

if len(feature_cols) == 0:
    raise ValueError("Tidak ada numeric feature yang tersedia untuk edge features.")

Num feature cols: 9
['block_number', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'month']


In [8]:
# =========================
# FEATURE SCALING
# =========================
# Fit scaler hanya di train, lalu transform val/test.

scaler = StandardScaler()

X_train_raw = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32).values
X_val_raw   = val_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32).values
X_test_raw  = test_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32).values

X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val   = scaler.transform(X_val_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


X_train: (35802, 9)
X_val  : (149079, 9)
X_test : (298160, 9)


In [9]:
# =========================
# DATASET & DATALOADER
# =========================

class EdgeDataset(Dataset):
    def __init__(self, df, features):
        self.source, self.target = map_nodes(df)
        self.edge_features = features
        self.labels = df[LABEL_COL].astype(np.float32).values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.source[idx], dtype=torch.long),
            torch.tensor(self.target[idx], dtype=torch.long),
            torch.tensor(self.edge_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

BATCH_SIZE = 1024

train_dataset = EdgeDataset(train_df, X_train)
val_dataset   = EdgeDataset(val_df, X_val)
test_dataset  = EdgeDataset(test_df, X_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

edge_feat_dim = X_train.shape[1]
print("edge_feat_dim:", edge_feat_dim)


edge_feat_dim: 9


In [10]:
# =========================
# MODEL
# =========================

class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)
        input_dim = embedding_dim * 2 + edge_feat_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        dst_emb = self.node_embedding(target)
        x = torch.cat([src_emb, dst_emb, edge_feat], dim=1)
        return self.mlp(x).squeeze(-1)

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128,
    dropout=0.3,
).to(DEVICE)

model


GraphEdgeClassifier(
  (node_embedding): Embedding(126505, 64)
  (mlp): Sequential(
    (0): Linear(in_features=137, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [11]:
# =========================
# PLAIN LOSS
# =========================
# Versi plain: tidak memakai class weight / pos_weight.
# Tujuannya sebagai baseline pembanding terhadap class weight dan focal loss.

labels = train_df[LABEL_COL].astype(int).values
num_positive = int(np.sum(labels == 1))
num_negative = int(np.sum(labels == 0))
raw_pos_weight = num_negative / max(num_positive, 1)

print("Train positives:", num_positive)
print("Train negatives:", num_negative)
print("Raw imbalance ratio negative/positive:", raw_pos_weight)
print("Loss mode: BCEWithLogitsLoss without pos_weight")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


Train positives: 702
Train negatives: 35100
Raw imbalance ratio negative/positive: 50.0
Loss mode: BCEWithLogitsLoss without pos_weight


In [12]:
# =========================
# TRAIN / EVALUATE FUNCTIONS
# =========================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()
        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * len(label)

    return total_loss / len(loader.dataset)


def predict_proba(model, loader):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for source, target, edge_feat, label in loader:
            source = source.to(DEVICE)
            target = target.to(DEVICE)
            edge_feat = edge_feat.to(DEVICE)

            logits = model(source, target, edge_feat)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(label.numpy())

    return np.concatenate(all_labels), np.concatenate(all_probs)


def compute_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])

    try:
        roc_auc = roc_auc_score(labels, probs)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(labels, probs)
    except ValueError:
        pr_auc = np.nan

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "confusion_matrix": cm,
    }


In [13]:
# =========================
# THRESHOLD SELECTION ON VALIDATION
# =========================

def select_threshold_from_validation(labels, probs, strategy="best_f1", recall_target=0.8, max_fp=None):
    thresholds = np.unique(np.quantile(probs, np.linspace(0, 1, 1001)))
    thresholds = np.unique(np.concatenate([thresholds, np.array([0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])]))

    rows = []
    for t in thresholds:
        m = compute_metrics(labels, probs, threshold=float(t))
        cm = m["confusion_matrix"]
        rows.append({
            "threshold": float(t),
            "accuracy": m["accuracy"],
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "tn": int(cm[0, 0]),
            "fp": int(cm[0, 1]),
            "fn": int(cm[1, 0]),
            "tp": int(cm[1, 1]),
        })

    df = pd.DataFrame(rows)

    if strategy == "recall_target":
        candidate = df[df["recall"] >= recall_target].copy()
        if max_fp is not None:
            candidate = candidate[candidate["fp"] <= max_fp]
        if len(candidate) > 0:
            # Dari threshold yang memenuhi recall, pilih FP paling kecil, lalu precision paling tinggi.
            best = candidate.sort_values(["fp", "precision", "f1"], ascending=[True, False, False]).iloc[0]
        else:
            best = df.sort_values(["f1", "recall"], ascending=False).iloc[0]
    elif strategy == "max_f1":
        best = df.sort_values(["f1", "recall"], ascending=False).iloc[0]
    else:
        best = df.sort_values(["f1", "recall"], ascending=False).iloc[0]

    return float(best["threshold"]), df.sort_values("f1", ascending=False)


In [14]:
# =========================
# TRAINING WITH VALIDATION MODEL SELECTION
# =========================

EPOCHS = 20
PATIENCE = 5
BEST_MODEL_PATH = OUTPUT_MODELS_DIR / "graph_plain_valsplit_best.pt"

history = []
best_val_pr_auc = -1
best_epoch = 0
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    val_labels, val_probs = predict_proba(model, val_loader)
    val_metrics = compute_metrics(val_labels, val_probs, threshold=0.5)

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy@0.5": val_metrics["accuracy"],
        "val_precision@0.5": val_metrics["precision"],
        "val_recall@0.5": val_metrics["recall"],
        "val_f1@0.5": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
    }
    history.append(row)

    print(row)

    score = val_metrics["pr_auc"] if not np.isnan(val_metrics["pr_auc"]) else -1
    if score > best_val_pr_auc:
        best_val_pr_auc = score
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_history.csv", index=False)
history_df


{'epoch': 1, 'train_loss': 0.29300425856644585, 'val_accuracy@0.5': 0.9996511916500648, 'val_precision@0.5': 0.0, 'val_recall@0.5': 0.0, 'val_f1@0.5': 0.0, 'val_roc_auc': 0.5873974308217768, 'val_pr_auc': 0.0005238466519448212}
{'epoch': 2, 'train_loss': 0.0949686809089835, 'val_accuracy@0.5': 0.9996511916500648, 'val_precision@0.5': 0.0, 'val_recall@0.5': 0.0, 'val_f1@0.5': 0.0, 'val_roc_auc': 0.5732186630094391, 'val_pr_auc': 0.0006380541848039489}
{'epoch': 3, 'train_loss': 0.0853398860801126, 'val_accuracy@0.5': 0.9996511916500648, 'val_precision@0.5': 0.0, 'val_recall@0.5': 0.0, 'val_f1@0.5': 0.0, 'val_roc_auc': 0.5714414940813513, 'val_pr_auc': 0.0006243740879065923}
{'epoch': 4, 'train_loss': 0.07926286172133626, 'val_accuracy@0.5': 0.9996511916500648, 'val_precision@0.5': 0.0, 'val_recall@0.5': 0.0, 'val_f1@0.5': 0.0, 'val_roc_auc': 0.5828257244040961, 'val_pr_auc': 0.0007379849538778852}
{'epoch': 5, 'train_loss': 0.07449414524663914, 'val_accuracy@0.5': 0.9996511916500648, 'v

,epoch,train_loss,val_accuracy@0.5,val_precision@0.5,val_recall@0.5,val_f1@0.5,val_roc_auc,val_pr_auc
0,1,0.293004,0.999651,0.000000,0.000000,0.000000,0.587397,0.000524
1,2,0.094969,0.999651,0.000000,0.000000,0.000000,0.573219,0.000638
2,3,0.085340,0.999651,0.000000,0.000000,0.000000,0.571441,0.000624
3,4,0.079263,0.999651,0.000000,0.000000,0.000000,0.582826,0.000738
4,5,0.074494,0.999651,0.000000,0.000000,0.000000,0.580742,0.000781
5,6,0.069411,0.999651,0.000000,0.000000,0.000000,0.589401,0.000838
6,7,0.064568,0.999651,0.000000,0.000000,0.000000,0.598213,0.000937
7,8,0.060097,0.999651,0.000000,0.000000,0.000000,0.605032,0.000966
8,9,0.056317,0.999651,0.000000,0.000000,0.000000,0.614366,0.001060
9,10,0.051168,0.999651,0.000000,0.000000,0.000000,0.619220,0.001094


In [15]:
# =========================
# LOAD BEST MODEL + SELECT THRESHOLD ON VALIDATION
# =========================

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))

val_labels, val_probs = predict_proba(model, val_loader)

# Strategi utama: target recall di validation, tapi cari FP terendah.
# Kamu bisa ubah recall_target menjadi 0.7 / 0.8 / 0.9 sesuai kebutuhan eksperimen.
BEST_THRESHOLD, val_threshold_table = select_threshold_from_validation(
    val_labels,
    val_probs,
    strategy="recall_target",
    recall_target=0.8,
    max_fp=None,
)

print("Best epoch:", best_epoch)
print("Best threshold from validation:", BEST_THRESHOLD)

val_threshold_table.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_thresholds_validation.csv", index=False)
val_threshold_table.head(20)


Best epoch: 20
Best threshold from validation: 9.955421634003868e-06


,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
1009,0.612406,0.998672,0.013333,0.038462,0.019802,148879,148,50,2
1008,0.542486,0.997686,0.010033,0.057692,0.017094,148731,296,49,3
1007,0.500000,0.996820,0.007009,0.057692,0.012500,148602,425,49,3
1006,0.493581,0.996686,0.006696,0.057692,0.012000,148582,445,49,3
1005,0.451171,0.995687,0.005025,0.057692,0.009245,148433,594,49,3
1002,0.390120,0.993701,0.004469,0.076923,0.008448,148136,891,48,4
1000,0.342787,0.991716,0.004191,0.096154,0.008032,147839,1188,47,5
1004,0.415808,0.994687,0.004021,0.057692,0.007519,148284,743,49,3
1001,0.364119,0.992702,0.003831,0.076923,0.007299,147987,1040,48,4
999,0.323953,0.990716,0.003726,0.096154,0.007174,147690,1337,47,5


In [16]:
# =========================
# FINAL TEST EVALUATION
# =========================

test_labels, test_probs = predict_proba(model, test_loader)
final_metrics = compute_metrics(test_labels, test_probs, threshold=BEST_THRESHOLD)
cm = final_metrics["confusion_matrix"]

final_result = {
    "model": "graph_plain_valsplit",
    "best_epoch_from_validation": best_epoch,
    "best_threshold_from_validation": BEST_THRESHOLD,
    "threshold_strategy": "recall_target_0.8_min_fp_on_validation",
    "threshold": BEST_THRESHOLD,
    "accuracy": final_metrics["accuracy"],
    "precision": final_metrics["precision"],
    "recall": final_metrics["recall"],
    "f1": final_metrics["f1"],
    "roc_auc": final_metrics["roc_auc"],
    "pr_auc": final_metrics["pr_auc"],
    "tn": int(cm[0, 0]),
    "fp": int(cm[0, 1]),
    "fn": int(cm[1, 0]),
    "tp": int(cm[1, 1]),
}

pd.DataFrame([final_result]).to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_final.csv", index=False)

print(json.dumps(final_result, indent=2))
print("Confusion Matrix:")
print(cm)


{
  "model": "graph_plain_valsplit",
  "best_epoch_from_validation": 20,
  "best_threshold_from_validation": 9.955421634003868e-06,
  "threshold_strategy": "recall_target_0.8_min_fp_on_validation",
  "threshold": 9.955421634003868e-06,
  "accuracy": 0.3336698416957338,
  "precision": 0.00028180212458673215,
  "recall": 0.875,
  "f1": 0.0005634227934703322,
  "roc_auc": 0.6475769865244754,
  "pr_auc": 0.00037697994446690925,
  "tn": 99431,
  "fp": 198665,
  "fn": 8,
  "tp": 56
}
Confusion Matrix:
[[ 99431 198665]
 [     8     56]]


In [17]:
# =========================
# THRESHOLD TABLE ON TEST - ANALYSIS ONLY
# =========================
# Ini hanya untuk analisis laporan, bukan untuk memilih threshold final.

test_threshold_rows = []
for t in [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, BEST_THRESHOLD]:
    m = compute_metrics(test_labels, test_probs, threshold=float(t))
    cm = m["confusion_matrix"]
    test_threshold_rows.append({
        "threshold": float(t),
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "roc_auc": m["roc_auc"],
        "pr_auc": m["pr_auc"],
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    })

test_threshold_df = pd.DataFrame(test_threshold_rows).drop_duplicates("threshold")
test_threshold_df.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_thresholds_test_analysis.csv", index=False)
test_threshold_df.sort_values("threshold")


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
10,0.00001,0.333670,0.000282,0.875000,0.000563,0.647577,0.000377,99431,198665,8,56
0,0.00100,0.684827,0.000351,0.515625,0.000702,0.647577,0.000377,204155,93941,31,33
1,0.00500,0.804578,0.000326,0.296875,0.000652,0.647577,0.000377,239874,58222,45,19
2,0.01000,0.849557,0.000357,0.250000,0.000713,0.647577,0.000377,253288,44808,48,16
3,0.02000,0.888929,0.000302,0.156250,0.000604,0.647577,0.000377,265033,33063,54,10
4,0.05000,0.932486,0.000349,0.109375,0.000695,0.647577,0.000377,278023,20073,57,7
5,0.10000,0.958713,0.000489,0.093750,0.000974,0.647577,0.000377,285844,12252,58,6
6,0.20000,0.979001,0.000484,0.046875,0.000957,0.647577,0.000377,291896,6200,61,3
7,0.30000,0.988275,0.000582,0.031250,0.001143,0.647577,0.000377,294662,3434,62,2
8,0.40000,0.993363,0.001042,0.031250,0.002017,0.647577,0.000377,296179,1917,62,2


In [18]:
# =========================
# STAGE 2: PREDICTION RESULT TABLE
# =========================

test_results = pd.DataFrame({
    "true_label": test_labels.astype(int),
    "probability": test_probs,
    "prediction": (test_probs >= BEST_THRESHOLD).astype(int),
})

test_with_pred = pd.concat([test_df.reset_index(drop=True), test_results], axis=1)
candidate_fraud_full = test_with_pred[test_with_pred["prediction"] == 1].copy()

print("test_with_pred:", test_with_pred.shape)
print("candidate_fraud_full:", candidate_fraud_full.shape)
print("Total real fraud in test:", int(test_with_pred["true_label"].sum()))

test_with_pred.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_test_predictions.csv", index=False)
candidate_fraud_full.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_candidate_fraud.csv", index=False)


test_with_pred: (298160, 28)
candidate_fraud_full: (198721, 28)
Total real fraud in test: 64


In [19]:
# =========================
# STAGE 2: RULE-BASED FILTERING ANALYSIS
# =========================

filter_results = []
total_real_fraud = int(test_with_pred["true_label"].sum())

rules = [("model_only", candidate_fraud_full)]

if "wash_score" in candidate_fraud_full.columns:
    rules.extend([
        ("wash_score >= 1", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 1]),
        ("wash_score >= 2", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 2]),
        ("wash_score >= 3", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 3]),
    ])

if "confidence_category" in candidate_fraud_full.columns:
    rules.extend([
        ("confidence medium/high", candidate_fraud_full[candidate_fraud_full["confidence_category"].isin(["medium", "high"])]),
        ("confidence high", candidate_fraud_full[candidate_fraud_full["confidence_category"] == "high"]),
    ])

for rule_name, filtered_df in rules:
    tp = int((filtered_df["true_label"] == 1).sum())
    fp = int((filtered_df["true_label"] == 0).sum())
    fn = total_real_fraud - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    filter_results.append({
        "rule": rule_name,
        "candidate_count": len(filtered_df),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

filter_df = pd.DataFrame(filter_results)
filter_df.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_stage2_filtering.csv", index=False)
filter_df


,rule,candidate_count,tp,fp,fn,precision,recall,f1
0,model_only,198721,56,198665,8,0.000282,0.8750,0.000563
1,wash_score >= 1,56,56,0,8,1.000000,0.8750,0.933333
2,wash_score >= 2,56,56,0,8,1.000000,0.8750,0.933333
3,wash_score >= 3,4,4,0,60,1.000000,0.0625,0.117647
4,confidence medium/high,0,0,0,64,0.000000,0.0000,0.000000
5,confidence high,0,0,0,64,0.000000,0.0000,0.000000


In [20]:
# =========================
# TOP-K RANKING + LIFT VS RANDOM
# =========================

topk_results = []
total_real_fraud = int(test_with_pred["true_label"].sum())
total_test = len(test_with_pred)
fraud_rate = total_real_fraud / total_test if total_test > 0 else 0

k_values = [100, 500, 1000, 5000, 10000, 20000, 50000]
k_values = [k for k in k_values if k <= total_test]

ranked = test_with_pred.sort_values("probability", ascending=False).reset_index(drop=True)

for k in k_values:
    topk = ranked.head(k)
    tp = int((topk["true_label"] == 1).sum())
    fp = int((topk["true_label"] == 0).sum())
    fn = total_real_fraud - tp
    precision = tp / k if k > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    expected_random_tp = k * fraud_rate
    lift = tp / expected_random_tp if expected_random_tp > 0 else 0

    topk_results.append({
        "top_k": k,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "expected_random_tp": expected_random_tp,
        "lift_vs_random": lift,
    })

topk_df = pd.DataFrame(topk_results)
topk_df.to_csv(OUTPUT_METRICS_DIR / "graph_plain_valsplit_topk_lift.csv", index=False)
topk_df


,top_k,tp,fp,fn,precision,recall,f1,expected_random_tp,lift_vs_random
0,100,0,100,64,0.00000,0.000000,0.000000,0.021465,0.000000
1,500,0,500,64,0.00000,0.000000,0.000000,0.107325,0.000000
2,1000,1,999,63,0.00100,0.015625,0.001880,0.214650,4.658750
3,5000,3,4997,61,0.00060,0.046875,0.001185,1.073249,2.795250
4,10000,5,9995,59,0.00050,0.078125,0.000994,2.146499,2.329375
5,20000,7,19993,57,0.00035,0.109375,0.000698,4.292997,1.630563
6,50000,18,49982,46,0.00036,0.281250,0.000719,10.732493,1.677150
